# 🏒 scrapernhl — API Reference & Testing Notebook

A comprehensive, runnable reference for every public method in the `scrapernhl` library.  
Each section demonstrates a function with live output you can inspect and re-run.

---

**Leagues supported:** `nhl` · `ahl` · `pwhl` · `ohl` · `whl` · `qmjhl`

| Section | Description |
|---|---|
| [1 — Setup](#1) | Imports & scraper initialisation |
| [2 — Bootstrap Accessors](#2) | Teams, seasons, conferences, divisions |
| [3 — Play-by-Play](#3) | `play_by_play()` · `scrape_multiple_games()` |
| [4 — Player Stats](#4) | `player_stats()` · `scrape_skaters()` · `scrape_goalies()` |
| [5 — Schedule](#5) | `schedule()` |
| [6 — Roster](#6) | `roster()` |
| [7 — Standings](#7) | `standings()` |
| [8 — Teams & Seasons](#8) | `teams_by_season()` · `seasons()` |
| [9 — Player Profile](#9) | `player_profile()` (non-NHL) |
| [10 — Bootstrap](#10) | `bootstrap()` (non-NHL) |
| [11 — URL & Raw](#11) | `url_for()` · `fetch_raw()` |
| [12 — NHL Analytics](#12) | `on_ice_stats()` · `team_strength_aggregates()` · `goal_replay()` |
| [13 — NHL-Only Methods](#13) | `scrape_game()` · `shifts()` · `shift_chart()` |
| [14 — Functional API](#14) | `scrape()` one-liner |
| [15 — Strength Notation](#15) | Strength / situation code reference |
| [16 — CLI Reference](#16) | Command-line usage |

---
## 1 — Setup <a id="1"></a>

Import dependencies and initialise one scraper per league.  
Non-NHL scrapers automatically fetch bootstrap/config data on init.

In [62]:
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from scrapernhl import HockeyScraper

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [63]:
# ── Initialise scrapers ────────────────────────────────────────────────────────
nhl   = HockeyScraper("nhl")
ahl   = HockeyScraper("ahl")
pwhl  = HockeyScraper("pwhl")
ohl   = HockeyScraper("ohl")
whl   = HockeyScraper("whl")
qmjhl = HockeyScraper("qmjhl")

print("Scrapers ready.")
print(f"  NHL config: {nhl.config}")
print(f"  AHL config: {ahl.config}")

Scrapers ready.
  NHL config: LeagueConfig(name='NHL', client_code=None, api_key=None, league_id=None, site_id=None, default_season=20232024, base_url='https://api-web.nhle.com', pbp_style='nhl', canvas_size=(200, 85), rate_limit_calls=100, rate_limit_period=1.0, ot_period_length=1200, ot_period_length_history=())
  AHL config: LeagueConfig(name='AHL', client_code='ahl', api_key='ccb91f29d6744675', league_id=4, site_id=3, default_season=90, base_url='https://lscluster.hockeytech.com/feed/index.php', pbp_style='hockeytech_a', canvas_size=(850, 400), rate_limit_calls=2, rate_limit_period=1.0, ot_period_length=300, ot_period_length_history=())


---
## 2 — Bootstrap Properties & Accessors <a id="2"></a>

Available on all **non-NHL** scrapers. Bootstrap data is fetched automatically on init and stored in `scraper.bootstrap_data`.

| Property / Method | Returns |
|---|---|
| `scraper.teams` | `list[dict]` — all teams |
| `scraper.current_season_id` | `str` — current season ID |
| `scraper.current_league_id` | `str` — current league ID |
| `get_teams(include_all)` | `list[dict]` |
| `get_team_by_id(id)` | `dict | None` |
| `get_team_by_code(code)` | `dict | None` |
| `get_seasons(season_type)` | `list[dict]` |
| `get_current_season()` | `dict | None` |
| `get_conferences(include_all)` | `list[dict]` |
| `get_divisions(include_all)` | `list[dict]` |

In [4]:
# ── Properties: teams, current_season_id, current_league_id ──────────────────
print("Current AHL season ID :", ahl.current_season_id)
print("Current AHL league ID :", ahl.current_league_id)
print("Number of AHL teams   :", len(ahl.teams))
print("First team entry      :", ahl.teams[0])

Current AHL season ID : 90
Current AHL league ID : 4
Number of AHL teams   : 32
First team entry      : {'id': '440', 'name': 'Abbotsford Canucks', 'nickname': 'Canucks', 'team_code': 'ABB', 'division_id': '25', 'logo': 'https://assets.leaguestat.com/ahl/logos/50x50/440.png'}


In [5]:
# ── get_teams() ────────────────────────────────────────────────────────────────
teams = ahl.get_teams()
print(f"get_teams() → {len(teams)} teams")
for t in teams[:5]:
    print(f"  id={t["id"]:>4}  code={t.get("team_code","?"):>5}  name={t["name"]}")

get_teams() → 32 teams
  id= 440  code=  ABB  name=Abbotsford Canucks
  id= 402  code=  BAK  name=Bakersfield Condors
  id= 413  code=  BEL  name=Belleville Senators
  id= 317  code=  BRI  name=Bridgeport Islanders
  id= 444  code=  CGY  name=Calgary Wranglers


In [6]:
# ── get_team_by_id()  &  get_team_by_code() ────────────────────────────────────
first_id   = teams[0]["id"]
first_code = teams[0].get("team_code", "")
print("get_team_by_id  :", ahl.get_team_by_id(first_id))
print("get_team_by_code:", ahl.get_team_by_code(first_code))

get_team_by_id  : {'id': '440', 'name': 'Abbotsford Canucks', 'nickname': 'Canucks', 'team_code': 'ABB', 'division_id': '25', 'logo': 'https://assets.leaguestat.com/ahl/logos/50x50/440.png'}
get_team_by_code: {'id': '440', 'name': 'Abbotsford Canucks', 'nickname': 'Canucks', 'team_code': 'ABB', 'division_id': '25', 'logo': 'https://assets.leaguestat.com/ahl/logos/50x50/440.png'}


In [7]:
# ── get_seasons() ──────────────────────────────────────────────────────────────
all_seasons = ahl.get_seasons("all")
regular     = ahl.get_seasons("regular")
playoffs    = ahl.get_seasons("playoff")
print(f"All seasons    : {len(all_seasons)}")
print(f"Regular seasons: {len(regular)}")
print(f"Playoff seasons: {len(playoffs)}")
print("Last 3:", [(s.get("id"), s.get("name")) for s in all_seasons[-3:]])

All seasons    : 74
Regular seasons: 22
Playoff seasons: 20
Last 3: [('25', '1997 All-Star Game'), ('26', '1996 All-Star Game'), ('27', '1995 All-Star Game')]


In [8]:
# ── get_current_season() ───────────────────────────────────────────────────────
print("Current season:", ahl.get_current_season())

Current season: {'id': '90', 'name': '2025-26 Regular Season', 'default_sort': '', 'start_date': '2025-10-07', 'hide_in_standings': False}


In [9]:
# ── get_conferences()  &  get_divisions() ─────────────────────────────────────
conferences = ahl.get_conferences(include_all=False)
divisions   = ahl.get_divisions(include_all=False)
print("Conferences:")
for c in conferences: print(f"  {c}")
print("Divisions:")
for d in divisions:   print(f"  {d}")

Conferences:
  {'conference_id': '1', 'conference_name': 'Eastern Conference'}
  {'conference_id': '2', 'conference_name': 'Western Conference'}
Divisions:
  {'id': '15', 'name': 'Atlantic Division', 'conference_id': '1'}
  {'id': '16', 'name': 'North Division', 'conference_id': '1'}
  {'id': '24', 'name': 'Central Division', 'conference_id': '2'}
  {'id': '25', 'name': 'Pacific Division', 'conference_id': '2'}


---
## 3 — Play-by-Play <a id="3"></a>

### `play_by_play(game_id, *, nhlify, raw, season_id, is_playoff)`
**Aliases:** `scrape_pbp()`, `scrape_game_pbp()`

Returns one row per on-ice event. Works across all six leagues.

Key columns: `event` · `period` · `time` · `game_seconds` · `x` · `y` · `shot_distance_ft` · `shot_angle_deg` · `score_home` · `score_away`

In [10]:
# ── AHL play-by-play ───────────────────────────────────────────────────────────
pbp_ahl = ahl.play_by_play(1027781)
print("Shape:", pbp_ahl.shape)
print("Event counts:")
print(pbp_ahl["event"].value_counts())
pbp_ahl[["event","period","time","game_seconds","x","y"]].head()

Shape: (61, 65)
Event counts:
event
shot             47
penalty          10
goalie_change     3
goal              1
Name: count, dtype: int64


,event,period,time,game_seconds,x,y
0,goalie_change,1,0:00,0,NaN,NaN
1,goalie_change,1,0:00,0,NaN,NaN
2,penalty,1,1:38,98,NaN,NaN
3,shot,1,1:46,106,-53.65,-3.19
4,shot,1,2:03,123,9.88,-29.32


In [14]:
# ── NHL play-by-play ───────────────────────────────────────────────────────────
pbp_nhl = nhl.play_by_play(2023020001)
print("Shape:", pbp_nhl.shape)
print("Event counts:")
print(pbp_nhl["event"].value_counts())
pbp_nhl.head(5)
# 
# pbp_nhl[["event","period","time","game_seconds","x","y"]].head()

Shape: (328, 19)
Event counts:
event
faceoff            60
shot-on-goal       57
hit                45
stoppage           44
missed-shot        31
blocked-shot       30
takeaway           18
giveaway           15
penalty            10
goal                8
period-start        3
delayed-penalty     3
period-end          3
game-end            1
Name: count, dtype: int64


,eventId,periodDescriptor,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,details,pptReplayUrl,elapsedTime,time_seconds,event,score_home,score_away,league,game_id,scraped_at
0,102,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:00,20:00,1551,left,520,period-start,8,NaN,NaN,00:00,0,period-start,0,0,NHL,2023020001,2026-03-04T19:58:52.504813+00:00
1,101,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:00,20:00,1551,left,502,faceoff,9,"{'eventOwnerTeamId': 18, 'losingPlayerId': 847...",NaN,00:00,0,faceoff,0,0,NHL,2023020001,2026-03-04T19:58:52.504813+00:00
2,8,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:35,19:25,1551,left,516,stoppage,15,{'reason': 'icing'},NaN,00:35,35,stoppage,0,0,NHL,2023020001,2026-03-04T19:58:52.504813+00:00
3,103,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:35,19:25,1551,left,502,faceoff,17,"{'eventOwnerTeamId': 14, 'losingPlayerId': 847...",NaN,00:35,35,faceoff,0,0,NHL,2023020001,2026-03-04T19:58:52.504813+00:00
4,9,"{'number': 1, 'periodType': 'REG', 'maxRegulat...",00:48,19:12,1551,left,503,hit,20,"{'xCoord': 64, 'yCoord': 42, 'zoneCode': 'D', ...",NaN,00:48,48,hit,0,0,NHL,2023020001,2026-03-04T19:58:52.504813+00:00


In [15]:
# ── raw=True → returns dict ───────────────────────────────────────────────────
raw_pbp = ahl.play_by_play(1027781, raw=True)
print("Type:", type(raw_pbp))
print("Top-level keys:", list(raw_pbp[0]["details"].keys())[:10])

Type: <class 'list'>
Top-level keys: ['goalieComingIn', 'goalieGoingOut', 'team_id', 'period', 'time']


In [16]:
# ── nhlify=False → keep raw HockeyTech rows ──────────────────────────────────
pbp_raw_rows = ahl.play_by_play(1027781, nhlify=False)
print("nhlify=False shape:", pbp_raw_rows.shape)
print("nhlify=True  shape:", pbp_ahl.shape)

nhlify=False shape: (62, 65)
nhlify=True  shape: (61, 65)


### `scrape_multiple_games(game_ids, **kwargs)`

Scrapes PBP for a list of game IDs and concatenates into one DataFrame. Failed games are skipped.

In [17]:
# ── scrape_multiple_games() ────────────────────────────────────────────────────
multi_pbp = ahl.scrape_multiple_games([1027779, 1027781, 1027785])
print("Total rows:", len(multi_pbp))
print("Rows per game:")
print(multi_pbp["game_id"].value_counts())

Total rows: 210
Rows per game:
game_id
1027785    76
1027779    73
1027781    61
Name: count, dtype: int64


---
## 4 — Player Stats <a id="4"></a>

### `player_stats(season, team, position, raw, **filters)`
**Aliases:** `scrape_skaters()`, `scrape_goalies()`

- **NHL:** per-team only — iterate over all teams for league-wide stats.
- **Non-NHL:** `team="all"` returns league-wide results.

In [18]:
# ── AHL skaters — all teams ───────────────────────────────────────────────────
ahl_skaters = ahl.player_stats(season=90, position="skaters")
print("Shape:", ahl_skaters.shape)
print("Top 10 scorers:")
ahl_skaters[["name","teamCode","GP","G","A","PTS"]].head(10)

Shape: (1099, 32)
Top 10 scorers:


,name,teamCode,GP,G,A,PTS
0,Jakob Pelletier,SYR,47,21,36,57
1,Laurent Dauphin,LAV,49,15,41,56
2,Alex Barré-Boulet,COL,52,17,37,54
3,Arthur Kaliyev,BEL,54,30,21,51
4,Alex Belzile,LAV,53,26,25,51
5,Seth Griffith,BAK,55,15,36,51
6,Conor Geekie,SYR,47,14,37,51
7,Cameron Hughes,TEX,52,13,38,51
8,Benoit-Olivier Groulx,TOR,53,27,23,50
9,Justin Robidas,CHI,48,22,28,50


In [19]:
# ── AHL goalies ────────────────────────────────────────────────────────────────
ahl_goalies = ahl.player_stats(season=90, position="goalies")
print("Shape:", ahl_goalies.shape)
ahl_goalies.head(5)

Shape: (123, 33)


,player_id,rookie,name,active,teamCode,GP,minutes_played,saves,S,save_percentage,goals_against,shutouts,wins,losses,ot_losses,shootout_goals_against,shootout_attempts,G,A,PIM,shootout_percentage,goals_against_average,rank,firstName,lastName,playerName,teamId,teamLogo,season,league,seasonName,seasonStartYear,seasonEndYear
0,9653,1,Nolan Lalonde,0,CLE,1,1:00,2,2,1.000,0,0,0,0,0,0,0,0,0,0,0.000,0.00,1,Nolan,Lalonde,Nolan Lalonde,373,https://assets.leaguestat.com/ahl/logos/50x50/...,90,ahl,2025-26 Regular Season,2025,2026
1,10102,1,Ethan Haider,0,MIL,1,60:00,22,23,0.957,1,0,1,0,0,0,0,0,0,0,0.000,1.00,2,Ethan,Haider,Ethan Haider,327,https://assets.leaguestat.com/ahl/logos/50x50/...,90,ahl,2025-26 Regular Season,2025,2026
2,9015,0,Pyotr Kochetkov,0,CHI,1,59:56,22,23,0.957,1,0,1,0,0,0,0,0,0,0,0.000,1.00,3,Pyotr,Kochetkov,Pyotr Kochetkov,330,https://assets.leaguestat.com/ahl/logos/50x50/...,90,ahl,2025-26 Regular Season,2025,2026
3,7105,0,Kyle Keyser,0,COL,11,642:41,263,279,0.943,16,2,8,1,1,0,0,0,0,0,0.000,1.49,4,Kyle,Keyser,Kyle Keyser,419,https://assets.leaguestat.com/ahl/logos/50x50/...,90,ahl,2025-26 Regular Season,2025,2026
4,7784,0,Kevin Mandolese,1,SYR,2,120:00,42,45,0.933,3,0,2,0,0,0,0,0,0,0,0.000,1.50,5,Kevin,Mandolese,Kevin Mandolese,324,https://assets.leaguestat.com/ahl/logos/50x50/...,90,ahl,2025-26 Regular Season,2025,2026


In [20]:
# ── scrape_skaters() alias ─────────────────────────────────────────────────────
print("scrape_skaters() shape:", ahl.scrape_skaters(season=90).shape)

scrape_skaters() shape: (1099, 32)


In [21]:
# ── scrape_goalies() alias ─────────────────────────────────────────────────────
print("scrape_goalies() shape:", ahl.scrape_goalies(season=90).shape)

scrape_goalies() shape: (123, 33)


In [22]:
# ── NHL skaters — per-team only ────────────────────────────────────────────────
mtl_skaters = nhl.player_stats(team="MTL", season=20232024, position="skaters")
print("MTL skaters shape:", mtl_skaters.shape)
mtl_skaters.head(5)

MTL skaters shape: (32, 22)


,playerId,headshot,positionCode,gamesPlayed,goals,assists,points,plusMinus,penaltyMinutes,powerPlayGoals,shorthandedGoals,gameWinningGoals,overtimeGoals,shots,shootingPctg,avgTimeOnIcePerGame,avgShiftsPerGame,faceoffWinPctg,firstName.default,lastName.default,lastName.cs,lastName.sk
0,8475233,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,D,60,6,18,24,-1,24,0,0,0,0,54,0.111111,1214.4833,22.0500,0.000000,David,Savard,NaN,NaN
1,8475848,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,R,77,16,15,31,-24,74,1,0,2,0,154,0.103896,827.4416,17.5974,0.411765,Brendan,Gallagher,NaN,NaN
2,8476469,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,R,66,17,8,25,4,34,0,2,3,0,139,0.122302,933.0909,19.3636,0.312500,Joel,Armia,NaN,NaN
3,8476871,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,L,54,5,8,13,-12,21,1,0,1,0,77,0.064935,776.3148,17.0556,0.227273,Tanner,Pearson,NaN,NaN
4,8476875,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,D,82,11,51,62,-24,58,5,2,0,0,187,0.058824,1533.1707,25.2927,0.000000,Mike,Matheson,NaN,NaN


---
## 5 — Schedule <a id="5"></a>

### `schedule(team, season, raw, **filters)`
**Alias:** `scrape_schedule()`

Use `team="all"` for the league-wide schedule (non-NHL), or a team code / numeric ID to filter.

In [23]:
# ── AHL full schedule ──────────────────────────────────────────────────────────
ahl_schedule = ahl.schedule(season=90)
print("Total games:", len(ahl_schedule))
ahl_schedule[["gameId","date","homeTeam","awayTeam","homeScore","awayScore","gameStatus"]].head(10)

Total games: 1152


,gameId,date,homeTeam,awayTeam,homeScore,awayScore,gameStatus
0,1027781,2025-10-10,Utica Comets,Cleveland Monsters,0,1,Final
1,1027779,2025-10-10,Rochester Americans,Toronto Marlies,4,3,Final
2,1027777,2025-10-10,Manitoba Moose,Laval Rocket,4,1,Final
3,1027780,2025-10-10,Texas Stars,Grand Rapids Griffins,3,4,Final
4,1027775,2025-10-10,Colorado Eagles,Calgary Wranglers,7,4,Final
5,1027778,2025-10-10,Ontario Reign,Tucson Roadrunners,4,5,Final OT
6,1027776,2025-10-10,Henderson Silver Knights,Abbotsford Canucks,1,2,Final OT
7,1027774,2025-10-10,Coachella Valley Firebirds,San Diego Gulls,0,5,Final
8,1027793,2025-10-11,Utica Comets,Cleveland Monsters,2,3,Final OT
9,1027792,2025-10-11,Toronto Marlies,Rochester Americans,4,1,Final


In [24]:
# ── NHL team schedule ──────────────────────────────────────────────────────────
mtl_schedule = nhl.schedule(team="MTL", season=20232024)
print("MTL games:", len(mtl_schedule))
mtl_schedule.head(5)

MTL games: 88


,id,season,gameType,gameDate,neutralSite,startTimeUTC,easternUTCOffset,venueUTCOffset,venueTimezone,gameState,gameScheduleState,tvBroadcasts,gameCenterLink,venue.default,awayTeam.id,awayTeam.commonName.default,awayTeam.placeName.default,awayTeam.placeNameWithPreposition.default,awayTeam.placeNameWithPreposition.fr,awayTeam.abbrev,awayTeam.logo,awayTeam.darkLogo,awayTeam.awaySplitSquad,awayTeam.hotelLink,awayTeam.hotelDesc,awayTeam.score,homeTeam.id,homeTeam.commonName.default,homeTeam.placeName.default,homeTeam.placeNameWithPreposition.default,homeTeam.placeNameWithPreposition.fr,homeTeam.abbrev,homeTeam.logo,homeTeam.darkLogo,homeTeam.homeSplitSquad,homeTeam.score,periodDescriptor.periodType,periodDescriptor.maxRegulationPeriods,gameOutcome.lastPeriodType,winningGoalie.playerId,winningGoalie.firstInitial.default,winningGoalie.lastName.default,winningGoalScorer.playerId,winningGoalScorer.firstInitial.default,winningGoalScorer.lastName.default,awayTeam.commonName.fr,venue.fr,homeTeam.commonName.fr,homeTeam.hotelLink,homeTeam.hotelDesc,threeMinRecap,threeMinRecapFr,condensedGame,awayTeam.airlineLink,awayTeam.airlineDesc,winningGoalie.lastName.cs,winningGoalie.lastName.sk,winningGoalScorer.lastName.cs,winningGoalScorer.lastName.fi,winningGoalScorer.lastName.sk,winningGoalie.lastName.fi,winningGoalie.lastName.sv,venue.es,winningGoalScorer.lastName.sv,awayTeam.placeName.fr,homeTeam.placeName.fr,homeTeam.airlineLink,homeTeam.airlineDesc
0,2023010018,20232024,1,2023-09-25,False,2023-09-25T23:00:00Z,-04:00,-04:00,America/Montreal,FINAL,OK,"[{'id': 33, 'market': 'H', 'countryCode': 'CA'...",/gamecenter/njd-vs-mtl/2023/09/25/2023010018,Centre Bell,1,Devils,New Jersey,New Jersey,du New Jersey,NJD,https://assets.nhle.com/logos/nhl/svg/NJD_ligh...,https://assets.nhle.com/logos/nhl/svg/NJD_dark...,True,https://njdevils.hotelplanner.com/evt/AEF5E16A...,Book Hotel,4,8,Canadiens,Montréal,Montréal,de Montréal,MTL,https://assets.nhle.com/logos/nhl/svg/MTL_ligh...,https://assets.nhle.com/logos/nhl/svg/MTL_dark...,False,2,REG,3,REG,8481033,A.,Schmid,8482110.0,D.,Mercer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023010034,20232024,1,2023-09-27,False,2023-09-27T23:00:00Z,-04:00,-04:00,America/Montreal,FINAL,OK,"[{'id': 294, 'market': 'A', 'countryCode': 'CA...",/gamecenter/ott-vs-mtl/2023/09/27/2023010034,Centre Bell,9,Senators,Ottawa,Ottawa,d'Ottawa,OTT,https://assets.nhle.com/logos/nhl/svg/OTT_ligh...,https://assets.nhle.com/logos/nhl/svg/OTT_dark...,False,NaN,NaN,3,8,Canadiens,Montréal,Montréal,de Montréal,MTL,https://assets.nhle.com/logos/nhl/svg/MTL_ligh...,https://assets.nhle.com/logos/nhl/svg/MTL_dark...,False,4,REG,3,REG,8480051,C.,Primeau,8481540.0,C.,Caufield,Sénateurs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023010051,20232024,1,2023-09-29,False,2023-09-29T23:00:00Z,-04:00,-04:00,America/Montreal,FINAL,OK,"[{'id': 131, 'market': 'H', 'countryCode': 'CA...",/gamecenter/tor-vs-mtl/2023/09/29/2023010051,Centre Bell,10,Maple Leafs,Toronto,Toronto,de Toronto,TOR,https://assets.nhle.com/logos/nhl/svg/TOR_ligh...,https://assets.nhle.com/logos/nhl/svg/TOR_dark...,False,NaN,NaN,2,8,Canadiens,Montréal,Montréal,de Montréal,MTL,https://assets.nhle.com/logos/nhl/svg/MTL_ligh...,https://assets.nhle.com/logos/nhl/svg/MTL_dark...,False,1,REG,3,REG,8478492,I.,Samsonov,8481614.0,M.,Kokkonen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023010061,20232024,1,2023-09-30,False,2023-09-30T23:00:00Z,-04:00,-04:00,America/Montreal,FINAL,OK,"[{'id': 324, 'market': 'N', 'countryCode': 'US...",/gamecenter/tor-vs-mtl/2023/09/30/2023010061,Centre Bell,10,Maple Leafs,Toronto,Toronto,de Toronto,TOR,https://assets.nhle.com/logos/nhl/svg/TOR_ligh...,https://assets.nhle.com/logos/nhl/svg/TOR_dark...,False,NaN,NaN,3,8,Canadiens,Montréal,Montréal,de Montréal,MTL,https://assets.nhle.com/logos/nhl/svg/MTL_ligh...,https://assets.

---
## 6 — Roster <a id="6"></a>

### `roster(team, season, raw)`
**Alias:** `scrape_roster()`

- **NHL:** tricode e.g. `"MTL"`
- **Non-NHL:** numeric team ID string — retrieve via `get_teams()`

In [25]:
# ── AHL roster ─────────────────────────────────────────────────────────────────
ahl_teams = ahl.get_teams()
team_id   = ahl_teams[0]["id"]
print(f"Roster for ID {team_id} — {ahl_teams[0]["name"]}")
ahl_roster = ahl.roster(team=team_id, season=90)
print("Shape:", ahl_roster.shape)
ahl_roster[["name","position","tp_jersey_number","birthdate","shoots"]].head(10)

Roster for ID 440 — Abbotsford Canucks
Shape: (28, 25)


,name,position,tp_jersey_number,birthdate,shoots
0,Danila Klimovich,RW,9,2003-01-09,R
1,Arshdeep Bains,LW,13,2001-01-09,L
2,Joseph LaBate,C,14,1993-04-16,L
3,Jujhar Khaira,C,15,1994-08-13,L
4,Dino Kambeitz,RW,17,2000-01-25,R
5,Mackenzie MacEachern,LW,20,1994-03-09,L
6,Chase Wouters,C,21,2000-02-08,R
7,Jonathan Lekkerimäki,RW,23,2004-07-24,R
8,Anri Ravinskis,LW,26,2003-01-02,L
9,Ben Berard,LW,29,1999-02-13,L


In [26]:
# ── OHL roster ─────────────────────────────────────────────────────────────────
ohl_teams = ohl.get_teams()
print("OHL teams sample:", [(t["id"], t["name"]) for t in ohl_teams[:3]])
ohl_roster = ohl.roster(team=ohl_teams[0]["id"])
print("Shape:", ohl_roster.shape)
ohl_roster[["name","position","tp_jersey_number"]].head(10)

OHL teams sample: [('7', 'Barrie Colts'), ('18', 'Brampton Steelheads'), ('1', 'Brantford Bulldogs')]
Shape: (27, 28)


,name,position,tp_jersey_number
0,Nicholas Desiderio,LW,2
1,William Schneid,RW,11
2,Calvin Crombie,RW,14
3,Joe Salandra,RW,15
4,Mason Zebeski,LW,19
5,Brad Gardiner,C,25
6,Emil Hemming,RW,28
7,Cole Beaudoin,C,29
8,Eamon Edgar,C,37
9,Ben Wilmott,C,53


In [27]:
# ── NHL roster ─────────────────────────────────────────────────────────────────
mtl_roster = nhl.roster(team="MTL", season=20232024)
print("Shape:", mtl_roster.shape)
mtl_roster.head(10)

Shape: (20, 19)


,id,headshot,sweaterNumber,positionCode,shootsCatches,heightInInches,weightInPounds,heightInCentimeters,weightInKilograms,birthDate,birthCountry,firstName.default,lastName.default,birthCity.default,birthStateProvince.default,birthCity.sv,birthCity.cs,birthCity.fi,birthCity.sk
0,8476981,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,17,R,R,75,224,191,102,1994-05-07,CAN,Josh,Anderson,Burlington,ON,NaN,NaN,NaN,NaN
1,8476469,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,40,R,R,75,216,191,98,1993-05-31,FIN,Joel,Armia,Pori,NaN,Björneborg,NaN,NaN,NaN
2,8481540,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,22,R,R,68,175,173,79,2001-01-02,USA,Cole,Caufield,Mosinee,WI,NaN,NaN,NaN,NaN
3,8481523,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,77,C,R,76,217,193,98,2001-01-21,CAN,Kirby,Dach,Fort Saskatchewan,AB,NaN,NaN,NaN,NaN
4,8477989,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,28,C,L,73,192,185,87,1996-02-02,USA,Christian,Dvorak,Palos,IL,NaN,NaN,NaN,NaN
5,8478133,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,71,C,R,73,187,185,85,1996-06-02,CAN,Jake,Evans,Toronto,ON,NaN,NaN,NaN,NaN
6,8475848,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,11,R,R,69,183,175,83,1992-05-06,CAN,Brendan,Gallagher,Edmonton,AB,NaN,NaN,NaN,NaN
7,8481093,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,49,L,L,69,184,175,83,1999-01-06,CAN,Rafael,Harvey-Pinard,Saguenay,QC,NaN,NaN,NaN,NaN
8,8481618,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,15,C,L,71,199,180,90,2001-01-28,CAN,Alex,Newhook,St. John's,NL,NaN,NaN,NaN,NaN
9,8479543,https://assets.nhle.com/mugs/nhl/20232024/MTL/...,55,L,L,73,217,185,98,1998-03-13,CAN,Michael,Pezzetta,Toronto,ON,NaN,NaN,NaN,NaN


---
## 7 — Standings <a id="7"></a>

### `standings(season, raw, **filters)`
**Alias:** `scrape_standings()`

HockeyTech leagues accept `context="home"` / `context="away"` for split standings.

In [28]:
# ── AHL overall standings ──────────────────────────────────────────────────────
ahl_standings = ahl.standings()
print("Shape:", ahl_standings.shape)
ahl_standings[["teamName","wins","losses","ot_losses","points","percentage","rank"]].head(10)

Shape: (32, 28)


,teamName,wins,losses,ot_losses,points,percentage,rank
0,Providence Bruins,41,10,1,83,0.798,1
1,Wilkes-Barre/Scranton Penguins,36,13,4,78,0.709,2
2,Charlotte Checkers,31,18,5,67,0.620,3
3,Hershey Bears,26,21,6,60,0.545,4
4,Lehigh Valley Phantoms,24,24,2,53,0.500,5
5,Bridgeport Islanders,22,24,3,52,0.481,6
6,Springfield Thunderbirds,21,25,5,49,0.462,7
7,Hartford Wolf Pack,20,26,4,46,0.442,8
8,Laval Rocket,34,17,2,73,0.652,1
9,Syracuse Crunch,32,17,3,68,0.642,2


In [29]:
# ── AHL home standings ─────────────────────────────────────────────────────────
ahl_home = ahl.standings(context="home")
ahl_home[["teamName","wins","losses","points"]].head(5)

,teamName,wins,losses,points
0,Providence Bruins,21,6,42
1,Wilkes-Barre/Scranton Penguins,15,7,34
2,Charlotte Checkers,14,10,30
3,Hershey Bears,13,12,32
4,Lehigh Valley Phantoms,12,12,27


In [30]:
# ── AHL away standings ─────────────────────────────────────────────────────────
ahl_away = ahl.standings(context="away")
ahl_away[["teamName","wins","losses","points"]].head(5)

,teamName,wins,losses,points
0,Providence Bruins,41,10,83
1,Wilkes-Barre/Scranton Penguins,36,13,78
2,Charlotte Checkers,31,18,67
3,Hershey Bears,26,21,60
4,Lehigh Valley Phantoms,24,24,53


In [31]:
# ── NHL standings ──────────────────────────────────────────────────────────────
nhl_standings = nhl.standings(season=20232024)
print("Shape:", nhl_standings.shape)
nhl_standings.head(5)

Shape: (32, 83)


,conferenceAbbrev,conferenceHomeSequence,conferenceL10Sequence,conferenceName,conferenceRoadSequence,conferenceSequence,date,divisionAbbrev,divisionHomeSequence,divisionL10Sequence,divisionName,divisionRoadSequence,divisionSequence,gameTypeId,gamesPlayed,goalDifferential,goalDifferentialPctg,goalAgainst,goalFor,goalsForPctg,homeGamesPlayed,homeGoalDifferential,homeGoalsAgainst,homeGoalsFor,homeLosses,homeOtLosses,homePoints,homeRegulationPlusOtWins,homeRegulationWins,homeTies,homeWins,l10GamesPlayed,l10GoalDifferential,l10GoalsAgainst,l10GoalsFor,l10Losses,l10OtLosses,l10Points,l10RegulationPlusOtWins,l10RegulationWins,l10Ties,l10Wins,leagueHomeSequence,leagueL10Sequence,leagueRoadSequence,leagueSequence,losses,otLosses,pointPctg,points,regulationPlusOtWinPctg,regulationPlusOtWins,regulationWinPctg,regulationWins,roadGamesPlayed,roadGoalDifferential,roadGoalsAgainst,roadGoalsFor,roadLosses,roadOtLosses,roadPoints,roadRegulationPlusOtWins,roadRegulationWins,roadTies,roadWins,seasonId,shootoutLosses,shootoutWins,streakCode,streakCount,teamLogo,ties,waiversSequence,wildcardSequence,winPctg,wins,placeName.default,teamName.default,teamName.fr,teamCommonName.default,teamAbbrev.default,placeName.fr,teamCommonName.fr
0,W,1,5,Western,2,1,2026-03-04,C,1,3,Central,2,1,2,60,81,1.350000,149,230,3.833333,30,50,72,122,4,4,48,22,22,0,22,10,5,27,32,4,0,12,6,6,0,6,1,12,2,1,10,9,0.758333,91,0.666667,40,0.633333,38,30,31,77,108,6,5,43,18,16,0,19,20252026,5,1,W,3,https://assets.nhle.com/logos/nhl/svg/COL_ligh...,0,32,0,0.683333,41,Colorado,Colorado Avalanche,Avalanche du Colorado,Avalanche,COL,NaN,NaN
1,W,4,1,Western,1,2,2026-03-04,C,3,1,Central,1,2,2,61,47,0.770492,165,212,3.475410,28,17,75,92,7,3,39,16,14,0,18,10,20,23,43,0,0,20,9,7,0,10,7,1,1,2,14,9,0.696721,85,0.557377,34,0.508197,31,33,30,90,120,7,6,46,18,17,0,20,20252026,3,4,W,10,https://assets.nhle.com/logos/nhl/svg/DAL_ligh...,0,31,0,0.622951,38,Dallas,Dallas Stars,Stars de Dallas,Stars,DAL,NaN,NaN
2,E,1,2,Eastern,7,1,2026-03-04,M,1,2,Metropolitan,3,1,2,60,37,0.616667,171,208,3.466667,33,28,100,128,8,2,48,22,17,0,23,10,9,26,35,1,2,16,7,6,0,7,2,3,12,3,16,6,0.683333,82,0.566667,34,0.466667,28,27,9,71,80,8,4,34,12,11,0,15,20252026,3,4,L,1,https://assets.nhle.com/logos/nhl/svg/CAR_ligh...,0,30,0,0.633333,38,Carolina,Carolina Hurricanes,Hurricanes de la Caroline,Hurricanes,CAR,Caroline,NaN
3,W,2,2,Western,3,3,2026-03-04,C,2,2,Central,3,3,2,62,29,0.467742,180,209,3.370968,32,8,95,103,7,7,43,14,10,0,18,10,11,30,41,2,1,15,6,4,0,7,4,7,4,4,16,10,0.661290,82,0.516129,32,0.370968,23,30,21,85,106,9,3,39,18,13,0,18,20252026,3,4,W,1,https://assets.nhle.com/logos/nhl/svg/MIN_ligh...,0,28,0,0.580645,36,Minnesota,Minnesota Wild,Wild du Minnesota,Wild,MIN,NaN,NaN
4,E,6,8,Eastern,1,2,2026-03-04,A,4,4,Atlantic,1,1,2,59,52,0.881356,158,210,3.559322,29,18,79,97,10,0,38,17,13,0,19,10,2,36,38,4,0,12,5,4,0,6,10,13,3,5,17,4,0.677966,80,0.576271,34,0.491525,29,30,34,79,113,7,4,42,17,16,0,19,20252026,2,4,L,3,https://assets.nhle.com/logos/nhl/svg/TBL_ligh...,0,29,0,0.644068,38,Tampa Bay,Tampa Bay Lightning,Lightning de Tampa Bay,Lightning,TBL,NaN,NaN


---
## 8 — Teams & Seasons <a id="8"></a>

### `teams_by_season(season, raw)`
Returns a DataFrame of teams that participated in a specific season.

### `seasons(season_type, raw)`
Options: `"all"`, `"regular"`, `"playoff"`

In [32]:
# ── teams_by_season() — NHL ────────────────────────────────────────────────────
nhl_teams = nhl.teams_by_season(season=20232024)
print("NHL teams shape:", nhl_teams.shape)
nhl_teams.head(5)

NHL teams shape: (32, 85)


,conferenceAbbrev,conferenceHomeSequence,conferenceL10Sequence,conferenceName,conferenceRoadSequence,conferenceSequence,date,divisionAbbrev,divisionHomeSequence,divisionL10Sequence,divisionName,divisionRoadSequence,divisionSequence,gameTypeId,gamesPlayed,goalDifferential,goalDifferentialPctg,goalAgainst,goalFor,goalsForPctg,homeGamesPlayed,homeGoalDifferential,homeGoalsAgainst,homeGoalsFor,homeLosses,homeOtLosses,homePoints,homeRegulationPlusOtWins,homeRegulationWins,homeTies,homeWins,l10GamesPlayed,l10GoalDifferential,l10GoalsAgainst,l10GoalsFor,l10Losses,l10OtLosses,l10Points,l10RegulationPlusOtWins,l10RegulationWins,l10Ties,l10Wins,leagueHomeSequence,leagueL10Sequence,leagueRoadSequence,leagueSequence,losses,otLosses,pointPctg,points,regulationPlusOtWinPctg,regulationPlusOtWins,regulationWinPctg,regulationWins,roadGamesPlayed,roadGoalDifferential,roadGoalsAgainst,roadGoalsFor,roadLosses,roadOtLosses,roadPoints,roadRegulationPlusOtWins,roadRegulationWins,roadTies,roadWins,seasonId,shootoutLosses,shootoutWins,streakCode,streakCount,teamLogo,ties,waiversSequence,wildcardSequence,winPctg,wins,placeName.default,teamName.default,teamName.fr,teamCommonName.default,teamAbbrev.default,placeName.fr,teamCommonName.fr,season,league
0,E,4,3,Eastern,1,1,2024-01-01,M,2,3,Metropolitan,1,1,2,35,26,0.742857,95,121,3.457143,16,13,46,59,4,0,24,11,10,0,12,10,14,25,39,3,0,14,7,5,0,7,9,9,2,1,9,1,0.728571,51,0.685714,24,0.571429,20,19,13,49,62,5,1,27,13,10,0,13,20232024,1,1,W,1,https://assets.nhle.com/logos/nhl/svg/NYR_ligh...,0,32,0,0.714286,25,NY Rangers,New York Rangers,Rangers de New York,Rangers,NYR,NaN,NaN,20232024,nhl
1,E,2,4,Eastern,3,2,2024-01-01,A,1,1,Atlantic,1,1,2,35,23,0.657143,91,114,3.257143,16,16,36,52,2,3,25,10,9,0,11,10,3,29,32,2,3,13,4,4,0,5,6,11,4,2,7,6,0.714286,50,0.571429,20,0.514286,18,19,7,55,62,5,3,25,10,9,0,11,20232024,0,2,W,3,https://assets.nhle.com/logos/nhl/svg/BOS_2023...,0,31,0,0.628571,22,Boston,Boston Bruins,Bruins de Boston,Bruins,BOS,NaN,NaN,20232024,nhl
2,W,3,2,Western,4,1,2024-01-01,P,2,1,Pacific,2,1,2,36,43,1.194444,93,136,3.777778,18,30,40,70,4,1,27,13,12,0,13,10,12,23,35,1,2,16,7,7,0,7,3,2,8,3,10,3,0.680556,49,0.638889,23,0.611111,22,18,13,53,66,6,2,22,10,10,0,10,20232024,1,0,L,1,https://assets.nhle.com/logos/nhl/svg/VAN_ligh...,0,27,0,0.638889,23,Vancouver,Vancouver Canucks,Canucks de Vancouver,Canucks,VAN,NaN,NaN,20232024,nhl
3,W,1,4,Western,8,2,2024-01-01,C,1,2,Central,4,1,2,37,23,0.621622,111,134,3.621622,19,31,51,82,4,0,30,15,15,0,15,10,11,29,40,2,1,15,7,7,0,7,1,4,15,4,11,3,0.662162,49,0.594595,22,0.594595,22,18,-8,60,52,7,3,19,7,7,0,8,20232024,1,1,W,2,https://assets.nhle.com/logos/nhl/svg/COL_ligh...,0,26,0,0.621622,23,Colorado,Colorado Avalanche,Avalanche du Colorado,Avalanche,COL,NaN,NaN,20232024,nhl
4,W,2,13,Western,5,3,2024-01-01,P,1,6,Pacific,3,2,2,38,21,0.552632,106,127,3.342105,18,23,41,64,3,2,28,10,9,0,13,10,-9,41,32,6,0,8,3,2,0,4,2,27,11,5,11,5,0.644737,49,0.473684,18,0.421053,16,20,-2,65,63,8,3,21,8,7,0,9,20232024,1,4,L,1,https://assets.nhle.com/logos/nhl/svg/VGK_ligh...,0,25,0,0.578947,22,Vegas,Vegas Golden Knights,Golden Knights de Vegas,Golden Knights,VGK,NaN,NaN,20232024,nhl


In [33]:
# ── teams_by_season() — AHL ────────────────────────────────────────────────────
ahl_tbs = ahl.teams_by_season(season=90)
print("AHL teams shape:", ahl_tbs.shape)
ahl_tbs[["name","id","season"]].head(5)

AHL teams shape: (32, 8)


,name,id,season
0,Abbotsford Canucks,440,90
1,Bakersfield Condors,402,90
2,Belleville Senators,413,90
3,Bridgeport Islanders,317,90
4,Calgary Wranglers,444,90


In [34]:
# ── seasons() — AHL ───────────────────────────────────────────────────────────
s_all = ahl.seasons("all")
s_reg = ahl.seasons("regular")
s_po  = ahl.seasons("playoff")
print("All     :", s_all.shape)
print("Regular :", s_reg.shape)
print("Playoffs:", s_po.shape)
s_all[["id","name"]].tail(5)

All     : (74, 6)
Regular : (22, 3)
Playoffs: (20, 3)


,id,name
69,23,1999 All-Star Game
70,24,1998 All-Star Game
71,25,1997 All-Star Game
72,26,1996 All-Star Game
73,27,1995 All-Star Game


In [35]:
# ── seasons() — NHL ───────────────────────────────────────────────────────────
nhl_seasons = nhl.seasons("regular")
print("NHL regular seasons:", nhl_seasons.shape)
nhl_seasons.tail(5)

NHL regular seasons: (108, 24)


,id,allStarGameInUse,conferencesInUse,divisionsInUse,endDate,entryDraftInUse,formattedSeasonId,minimumPlayoffMinutesForGoalieStatsLeaders,minimumRegularGamesForGoalieStatsLeaders,nhlStanleyCupOwner,numberOfGames,olympicsParticipation,pointForOTLossInUse,preseasonStartdate,regularSeasonEndDate,rowInUse,seasonOrdinal,startDate,supplementalDraftInUse,tiesInUse,totalPlayoffGames,totalRegularSeasonGames,wildcardInUse,league
103,19871988,1,1,1,1988-05-26T12:00:00,1,1987-88,240,24,1,80,0,0,None,1988-04-03T20:30:00,0,71,1987-10-08T12:00:00,1,1,83,840,0,nhl
104,19711972,1,0,1,1972-05-11T20:00:00,1,1971-72,120,23,1,78,0,0,None,1972-04-02T20:30:00,0,55,1971-10-08T20:00:00,0,1,36,546,0,nhl
105,19801981,1,1,1,1981-05-21T20:00:00,1,1980-81,120,24,1,80,0,0,None,1981-04-05T20:30:00,0,64,1980-10-09T20:00:00,0,1,68,840,0,nhl
106,20242025,0,1,1,2025-06-24T00:00:00,1,2024-25,240,25,1,82,0,1,2024-09-21T19:00:00,2025-04-17T21:30:00,1,107,2024-10-04T13:00:00,0,0,86,1312,1,nhl
107,20252026,0,1,1,2026-06-15T00:00:00,1,2025-26,30,19,1,82,1,1,2025-09-20T19:00:00,2026-04-17T00:00:00,1,108,2025-10-07T17:00:00,0,0,0,1312,1,nhl


---
## 9 — Player Profile <a id="9"></a>

### `player_profile(player_id, season, stats_type, raw)` — Non-NHL only

Returns a dict with keys: `info` · `careerStats` · `seasonStats` · `gameByGame` · `shotLocations`

In [36]:
# ── Grab a player ID from AHL skater stats ───────────────────────────────────
player_id   = int(ahl_skaters.iloc[0]["player_id"])
player_name = ahl_skaters.iloc[0]["name"]
print(f"Profile for: {player_name} (ID {player_id})")

Profile for: Jakob Pelletier (ID 8744)


In [37]:
# ── player_profile() ──────────────────────────────────────────────────────────
profile = ahl.player_profile(player_id, season=90)
print("Keys:", list(profile.keys()))
print("── Info ──")
print(profile["info"])

Keys: ['info', 'currentSeason', 'careerStats', 'currentSeasonStats', 'gameByGame', 'playerShots', 'media', 'seasons', 'playerProfileHeaders', 'playerProfileBioHeaders', 'showPlayerDraft', 'draftInfo', 'translations', 'display', 'svfLang']
── Info ──
{'jerseyNumber': '22', 'firstName': 'Jakob', 'lastName': 'Pelletier', 'playerId': '8744', 'personId': '8992', 'position': 'F', 'shoots': 'L', 'catches': 'L', 'height': '5-10', 'height_sans_hyphen': '5.10', 'height_hyphenated': '5-10', 'weight': '172', 'birthDate': '2001-03-07', 'profileImage': 'https://assets.leaguestat.com/ahl/240x240/8744.jpg', 'teamImage': 'https://assets.leaguestat.com/ahl/logos/324.png', 'bio': '', 'teamName': 'Syracuse Crunch', 'nickName': 'Crunch', 'division': 'North Division', 'drafts': [{'id': '111316', 'draft_league': 'NHL', 'draft_team': 'Calgary Flames', 'draft_team_id': '5', 'draft_year': '2019', 'draft_round': '1', 'draft_rank': '26', 'draft_junior_team': 'Moncton (QMJHL)', 'draft_logo': 'https://assets.league

In [38]:
# ── Career stats ──────────────────────────────────────────────────────────────
career = profile.get("careerStats", [])
print(f"Career stats ({len(career)} seasons):")
for s in career: print(s)

Career stats (1 seasons):
{'sections': [{'title': 'Regular Season', 'headers': {'season_name': {'properties': {'key': 'season_name', 'hidden': False, 'class': '', 'label': 'Season', 'title': 'Season', 'sortable': True, 'align': 'center', 'highlight': False, 'sortKey': ''}}, 'team_name': {'properties': {'key': 'team_name', 'hidden': False, 'class': '', 'label': 'Team', 'title': 'Team', 'sortable': True, 'align': 'center', 'highlight': False, 'sortKey': ''}}, 'games_played': {'properties': {'key': 'games_played', 'hidden': False, 'class': '', 'label': 'GP', 'title': 'Games Played', 'sortable': True, 'align': 'center', 'highlight': False, 'sortKey': ''}}, 'goals': {'properties': {'key': 'goals', 'hidden': False, 'class': '', 'label': 'G', 'title': 'Goals', 'sortable': True, 'align': 'center', 'highlight': False, 'sortKey': ''}}, 'assists': {'properties': {'key': 'assists', 'hidden': False, 'class': '', 'label': 'A', 'title': 'Assists', 'sortable': True, 'align': 'center', 'highlight': Fal

In [39]:
# ── Game-by-game log ──────────────────────────────────────────────────────────
gbg = profile.get("gameByGame", [])
print(f"Game-by-game: {len(gbg)} entries")
if gbg:
    display(pd.json_normalize(gbg[0]['sections'][0]['data']).head(5))

Game-by-game: 1 entries


,prop.game.gameLink,row.date_played,row.shots,row.goals,row.pp,row.sh,row.gw,row.plusminus,row.assists,row.shootout_goals,row.shootout_attempts,row.penalty_minutes,row.points,row.game
0,1027785,2025-10-11,8,2,2,0,0,-2,1,0,0,0,3,SYR @ HER
1,1027796,2025-10-12,2,0,0,0,0,0,1,0,0,0,1,SYR @ HER
2,1027824,2025-10-18,3,0,0,0,0,2,1,0,0,0,1,ROC @ SYR
3,1027827,2025-10-19,2,0,0,0,0,0,0,0,0,0,0,SYR @ BEL
4,1027835,2025-10-22,3,0,0,0,0,1,1,0,0,5,1,SYR @ ROC


---
## 10 — Bootstrap <a id="10"></a>

### `bootstrap(game_id, season, page_name, **filters)` — Non-NHL only

Explicitly fetches raw bootstrap/config data. Only needed when you require a specific game context or historical season — data is auto-fetched on init.

In [40]:
# ── Default bootstrap ─────────────────────────────────────────────────────────
bs = ahl.bootstrap()
print("Type:", type(bs))
print("Keys:", list(bs.keys())[:15])
print("current_season_id:", bs.get("current_season_id"))
print("Team count        :", len(bs.get("teamsNoAll", [])))

Type: <class 'dict'>
Keys: ['firebaseUrl', 'firebaseToken', 'firebaseApiKey', 'current_league_id', 'current_season_id', 'leagues', 'seasons', 'conferences', 'conferencesAll', 'divisions', 'divisionsAll', 'first_season_year', 'regularSeasons', 'playoffSeasons', 'teams']
current_season_id: 90
Team count        : 32


In [41]:
# ── Game-specific bootstrap ───────────────────────────────────────────────────
game_bs = ahl.bootstrap(game_id=1027781, page_name="gamecenter")
print("game_bootstrap keys:", list(game_bs.keys())[:10])

game_bootstrap keys: ['firebaseUrl', 'firebaseToken', 'firebaseApiKey', 'current_league_id', 'current_season_id', 'leagues', 'seasons', 'conferences', 'conferencesAll', 'divisions']


In [42]:
# ── Historical season ─────────────────────────────────────────────────────────
hist_bs = ahl.bootstrap(season="88")
print("Season 88 current_season_id:", hist_bs.get("current_season_id"))

Season 88 current_season_id: 90


---
## 11 — URL Inspection & Raw Fetch <a id="11"></a>

### `url_for(data_type, **kwargs)`
Returns the URL for a given endpoint **without making any HTTP request**. Good for debugging.

### `fetch_raw(data_type, **kwargs)`
Returns the unprocessed API response dict, bypassing all parsing.

Supported `data_type` values: `"pbp"` · `"stats"` · `"schedule"` · `"roster"` · `"standings"` · `"bootstrap"` · `"scorebar"` · `"player_profile"` · `"player_game_log"` (NHL only)

In [43]:
# ── url_for() — NHL ────────────────────────────────────────────────────────────
print("PBP      :", nhl.url_for("pbp",       game_id=2023020001))
print("stats    :", nhl.url_for("stats",     team="MTL", season=20232024))
print("schedule :", nhl.url_for("schedule",  team="MTL", season=20232024))
print("standings:", nhl.url_for("standings", season=20232024))
print("roster   :", nhl.url_for("roster",    team="MTL", season=20232024))

PBP      : https://api-web.nhle.com/v1/gamecenter/2023020001/play-by-play
stats    : https://api-web.nhle.com/v1/club-stats/MTL/20232024/2
schedule : https://api-web.nhle.com/v1/club-schedule-season/MTL/20232024
standings: https://api-web.nhle.com/v1/standings/2026-03-04
roster   : https://api-web.nhle.com/v1/roster/MTL/20232024


In [44]:
# ── url_for() — AHL ────────────────────────────────────────────────────────────
print("PBP            :", ahl.url_for("pbp",            game_id=1027781))
print("standings      :", ahl.url_for("standings",      season=90))
print("player_profile :", ahl.url_for("player_profile", player_id=player_id, season=90))

PBP            : https://lscluster.hockeytech.com/feed/index.php?feed=statviewfeed&view=gameCenterPlayByPlay&game_id=1027781&key=ccb91f29d6744675&client_code=ahl&league_id=4&fmt=json&lang=en
standings      : https://lscluster.hockeytech.com/feed/index.php?feed=statviewfeed&view=teams&season=90&key=ccb91f29d6744675&client_code=ahl&league_id=4&site_id=3&groupTeamsBy=division&context=overall&special=false
player_profile : https://lscluster.hockeytech.com/feed/index.php?feed=statviewfeed&view=player&key=ccb91f29d6744675&client_code=ahl&league_id=4&player_id=8744&season_id=90&site_id=3&lang=en&statsType=standard


In [45]:
# ── fetch_raw() — AHL standings ───────────────────────────────────────────────
raw_standings = ahl.fetch_raw("standings", season=90)
print("Type:", type(raw_standings))
print("Keys:", list(raw_standings[0]['sections'][0]['data'][0]['row'].keys())[:10])

Type: <class 'list'>
Keys: ['team_code', 'wins', 'losses', 'ot_losses', 'shootout_losses', 'regulation_wins', 'row', 'points', 'penalty_minutes', 'streak']


In [46]:
# ── fetch_raw() — AHL PBP ─────────────────────────────────────────────────────
raw_pbp_ahl = ahl.fetch_raw("pbp", game_id=1027781)
print("Type:", type(raw_pbp_ahl))
print("Keys:", list(raw_pbp_ahl)[:10])

Type: <class 'list'>
Keys: [{'event': 'goalie_change', 'details': {'goalieComingIn': {'id': 8860, 'firstName': 'Nico', 'lastName': 'Daws', 'jerseyNumber': 50, 'position': 'G', 'birthDate': '', 'playerImageURL': 'https://assets.leaguestat.com/ahl/120x160/8860.jpg'}, 'goalieGoingOut': None, 'team_id': '390', 'period': {'id': '1', 'shortName': '1', 'longName': '1st'}, 'time': '0:00'}}, {'event': 'goalie_change', 'details': {'goalieComingIn': {'id': 10893, 'firstName': 'Ivan', 'lastName': 'Fedotov', 'jerseyNumber': 28, 'position': 'G', 'birthDate': '', 'playerImageURL': 'https://assets.leaguestat.com/ahl/120x160/10893.jpg'}, 'goalieGoingOut': None, 'team_id': '373', 'period': {'id': '1', 'shortName': '1', 'longName': '1st'}, 'time': '0:00'}}, {'event': 'penalty', 'details': {'game_penalty_id': '306986', 'period': {'id': '1', 'shortName': '1', 'longName': '1st'}, 'time': '1:38', 'againstTeam': {'id': 373, 'name': '', 'city': '', 'nickname': '', 'abbreviation': 'CLE', 'logo': ''}, 'minutes':

In [47]:
# ── fetch_raw() — NHL PBP ─────────────────────────────────────────────────────
raw_pbp_nhl = nhl.fetch_raw("pbp", game_id=2023020001)
print("Type:", type(raw_pbp_nhl))
print("Keys:", list(raw_pbp_nhl.keys())[:10])

Type: <class 'dict'>
Keys: ['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue', 'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset']


---
## 12 — NHL Analytics Pipeline <a id="12"></a>

NHL-only methods for advanced analytics.

| Method | Description |
|---|---|
| `scrape_game(game_id)` | High-level game scrape — shifts + PBP merged |
| `on_ice_stats(pbp)` | Per-player, per-strength on-ice stats (Corsi, Fenwick, TOI) |
| `team_strength_aggregates(pbp)` | Team-level shot/goal stats by strength state |
| `goal_replay(url)` | Fetch goal replay sprite frames from a `pptReplayUrl` |
| `tracking_dict_to_df(frames)` | Convert replay frames → tidy DataFrame with rink coordinates |

In [48]:
# ── scrape_game() ─────────────────────────────────────────────────────────────
game = nhl.scrape_game(2023020001)
print("Type:", type(game))
if isinstance(game, pd.DataFrame):
    print("Shape:",   game.shape)
    print("Columns:", list(game.columns)[:15])
    display(game.head())

Type: <class 'pandas.core.frame.DataFrame'>
Shape: (1817, 108)
Columns: ['#', 'Per', 'strength', 'Time:Elapsed Game', 'Event', 'Description', 'Time', 'timeRemaining', 'timeInPeriodSec', 'timeRemainingSec', 'home_on_ice', 'away_on_ice', 'home_goalie', 'away_goalie', 'merge_idx']


,#,Per,strength,Time:Elapsed Game,Event,Description,Time,timeRemaining,timeInPeriodSec,timeRemainingSec,home_on_ice,away_on_ice,home_goalie,away_goalie,merge_idx,eventId,timeInPeriod,timeRemaining_api,situationCode,homeTeamDefendingSide,typeCode,event_api,sortOrder,period,periodType,maxRegulationPeriods,teamId_,losingPlayerId,winningPlayerId,xCoord,yCoord,zoneCode,reason,hittingPlayerId,hitteePlayerId,playerId,shotType,shootingPlayerId,goalieInNetId,awaySOG,homeSOG,blockingPlayerId,pptReplayUrl,scoringPlayerId,scoringPlayerTotal,assist1PlayerId,assist1PlayerTotal,assist2PlayerId,assist2PlayerTotal,awayScore,homeScore,highlightClipSharingUrl,highlightClip,secondaryReason,typeCode_1,descKey,duration,committedByPlayerId,drawnByPlayerId,highlightClipSharingUrlFr,highlightClipFr,isHome,eventTeam,html_event,home_on_id,away_on_id,homeGoalie_on_id,awayGoalie_on_id,home_on_full_name,away_on_full_name,homeGoalie_on_full_name,awayGoalie_on_full_name,home_on_count,away_on_count,homeGoalie_on_count,awayGoalie_on_count,n_home_skaters,n_away_skaters,pulled_home,pulled_away,home_strength,away_strength,gameStrength,detailedGameStrength,elapsedTime,player1Id,player2Id,player3Id,player1Name,player2Name,player3Name,gameId,homeTeam,awayTeam,teamId,isGoalie,shift_start_type,start_dot,start_zone,venue,venueLocation,gameDate,gameType,startTimeUTC,easternUTCOffset,venueUTCOffset,scrapedOn,source
0,1,1,,0:0020:00,PGSTR,,00:00,20:00,0,1200,[],[],[],[],0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,1,1,<NA>,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,NaN,NaN,NaN,2023020001,TBL,NSH,NaN,NaN,NaN,NaN,NaN,Amalie Arena,Tampa,2023-10-10,2,2023-10-10T21:30:00Z,-04:00,-04:00,2026-03-04T20:00:22.420523,NHL Play-by-Play API
1,2,1,,0:0020:00,PGEND,,00:00,20:00,0,1200,[],[],[],[],0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,1,1,<NA>,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,NaN,NaN,NaN,2023020001,TBL,NSH,NaN,NaN,NaN,NaN,NaN,Amalie Arena,Tampa,2023-10-10,2,2023-10-10T21:30:00Z,-04:00,-04:00,2026-03-04T20:00:22.420523,NHL Play-by-Play API
2,3,1,,0:0020:00,ANTHEM,,00:00,20:00,0,1200,[],[],[],[],0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,1,1,<NA>,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,NaN,NaN,NaN,2023020001,TBL,NSH,NaN,NaN,NaN,NaN,NaN,Amalie Arena,Tampa,2023-10-10,2,2023-10-10T21:30:00Z,-04:00,-04:00,2026-03-04T20:00:22.420523,NHL Play-by-Play API
3,4,1,,0:0020:00,PSTR,Period Start- Local time: 5:32 EDT,00:00,20:00,0,1200,"[64, 71, 38, 48, 77]","[75, 90, 9, 27, 45]",[31],[74],0.0,102.0,00:00,20:00,1551,left,520.0,period-start,8.0,1,REG,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NSH,PSTR,"[8477353, 8478519, 8479542, 8480246, 8475167]","[8481704, 8475158, 8476887, 8474151, 8478851]",[8477992],[8477424],"[Tyler Motte, Anthony Cirelli, Brandon Hagel, ...","[Juuso Parssinen, Ryan O'Reilly, Filip Forsber...",[Jonas Johansson],[Juuse Saros],5.0,5.0,1.0,1.0,4.0,4.0,0,0,5,5,5v5,5v5,0,<NA>,<NA>,<NA>,NaN,NaN,NaN,2023020001,TBL,NSH,NaN,NaN,NaN,NaN,NaN,Amalie Arena,Tampa,2023-10-10,2,2023-10-10T21:30:00Z,-04:00,-04:00,2026-03-04T20:00:22.420523,NHL Play-by-Play API
4,5,1,NaN,NaN,ON,NaN,0:00,20:00,0,1200,NaN,NaN,NaN,NaN,NaN,NaN,0:00,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,20:00,NaN,NaN,NaN,NaN,1.0,TBL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

`engineer_xg_features()` and `predict_xg()` have been **sunset** and removed from the public API. Use `on_ice_stats()` and `team_strength_aggregates()` directly on the PBP DataFrame. The `goal_replay()` and `tracking_dict_to_df()` methods remain available.

In [ ]:
# ── on_ice_stats() ───────────────────────────────────────────────────────────
player_stats = nhl.on_ice_stats(pbp_nhl, rates=True)
print("Shape:", player_stats.shape)
player_stats.head(5)

In [50]:
# # ── team_strength_aggregates() ───────────────────────────────────────────────
# strength_agg = nhl.team_strength_aggregates(pbp_xg)
# print("Type:", type(strength_agg))
# if isinstance(strength_agg, pd.DataFrame):
#     print("Shape:", strength_agg.shape)
#     strength_agg.head()

In [51]:
# ── goal_replay() ─────────────────────────────────────────────────────────────
goal_replay_url = pbp_nhl.query("typeDescKey=='goal'")['pptReplayUrl'].iloc[0]

replay = nhl.goal_replay('https://wsr.nhle.com/sprites/20232024/2023020001/ev154.json')
print("Type:", type(replay))
if isinstance(replay, list):
    print(f"{len(replay)} goal replay(s) found")
    for r in replay[:3]: print(r)
else:
    print(replay)

Type: <class 'list'>
120 goal replay(s) found
{'timeStamp': 16969742463, 'onIce': {'18002': {'id': 18002, 'playerId': 8474568, 'x': 1447.8958, 'y': 263.9861, 'sweaterNumber': 2, 'teamId': 18, 'teamAbbrev': 'NSH'}, '14091': {'id': 14091, 'playerId': 8474564, 'x': 1241.4901, 'y': 52.9191, 'sweaterNumber': 91, 'teamId': 14, 'teamAbbrev': 'TBL'}, '18003': {'id': 18003, 'playerId': 8478468, 'x': 1579.1407, 'y': 0.1292, 'sweaterNumber': 3, 'teamId': 18, 'teamAbbrev': 'NSH'}, '14048': {'id': 14048, 'playerId': 8480246, 'x': 256.8806, 'y': 616.6909, 'sweaterNumber': 48, 'teamId': 14, 'teamAbbrev': 'TBL'}, '14021': {'id': 14021, 'playerId': 8478010, 'x': 924.7064, 'y': 374.1189, 'sweaterNumber': 21, 'teamId': 14, 'teamAbbrev': 'TBL'}, '14077': {'id': 14077, 'playerId': 8475167, 'x': 545.2549, 'y': 77.4117, 'sweaterNumber': 77, 'teamId': 14, 'teamAbbrev': 'TBL'}, '18009': {'id': 18009, 'playerId': 8476887, 'x': 1232.7683, 'y': 229.5046, 'sweaterNumber': 9, 'teamId': 18, 'teamAbbrev': 'NSH'}, '18

In [52]:
from scrapernhl import tracking_dict_to_df

# Convert goal replay frames to a tidy DataFrame
# Each row = one entity (player or puck) at one timestamp
tracking_df = tracking_dict_to_df(replay)
print(f"Shape: {tracking_df.shape}")
print(f"Columns: {list(tracking_df.columns)}")
tracking_df.head(10)


Shape: (1581, 17)
Columns: ['entity_id', 'id', 'playerId', 'x', 'y', 'sweaterNumber', 'teamId', 'teamAbbrev', 'timeStamp', 'rink_x', 'rink_y', 'is_puck', 'frame', 'dx', 'dy', 'dt', 'speed']


,entity_id,id,playerId,x,y,sweaterNumber,teamId,teamAbbrev,timeStamp,rink_x,rink_y,is_puck,frame,dx,dy,dt,speed
0,1,1,NaN,964.1437,363.1916,,NaN,,16969742463,-19.654692,12.234033,True,0,NaN,NaN,NaN,NaN
1,1,1,NaN,985.2531,341.396,,NaN,,16969742464,-17.895575,14.050333,True,1,1.759117,1.8163,1.0,2.528525
2,1,1,NaN,1005.7219,334.1392,,NaN,,16969742465,-16.189842,14.655067,True,2,1.705733,0.604733,1.0,1.809759
3,1,1,NaN,1025.5052,330.1005,,NaN,,16969742466,-14.541233,14.991625,True,3,1.648608,0.336558,1.0,1.682611
4,1,1,NaN,1045.1238,326.034,,NaN,,16969742467,-12.90635,15.3305,True,4,1.634883,0.338875,1.0,1.669635
5,1,1,NaN,1076.0842,313.9086,,NaN,,16969742468,-10.326317,16.34095,True,5,2.580033,1.01045,1.0,2.770845
6,1,1,NaN,1127.8361,277.8365,,NaN,,16969742469,-6.013658,19.346958,True,6,4.312658,3.006008,1.0,5.25691
7,1,1,NaN,1177.7957,238.3239,,NaN,,16969742470,-1.850358,22.639675,True,7,4.1633,3.292717,1.0,5.308018
8,1,1,NaN,1226.3135,199.7532,,NaN,,16969742471,2.192792,25.8539,True,8,4.04315,3.214225,1.0,5.165104
9,1,1,NaN,1273.9906,161.8813,,NaN,,16969742472,6.165883,29.009892,True,9,3.973092,3.155992,1.0,5.074026


---
## 13 — NHL-Only Methods <a id="13"></a>

Methods only available on an NHL scraper. Extend beyond `play_by_play()` to include shift-level data.

| Method | Description |
|---|---|
| `scrape_game(game_id)` | Full game: shifts + PBP merged into one DataFrame |
| `shifts(game_id)` | Raw shift-level data for every player in a game |
| `shift_chart(game_id)` | Shift chart data formatted for visualisation |

In [59]:
# ── scrape_game() — shifts + PBP merged ──────────────────────────────────────
game_full = nhl.scrape_game(2023020001)
print('Type  :', type(game_full))
if isinstance(game_full, pd.DataFrame):
    print('Shape :', game_full.shape)
    print('Cols  :', list(game_full.columns)[:20])
    display(game_full.head())

Type  : <class 'pandas.core.frame.DataFrame'>
Shape : (1817, 108)
Cols  : ['#', 'Per', 'strength', 'Time:Elapsed Game', 'Event', 'Description', 'Time', 'timeRemaining', 'timeInPeriodSec', 'timeRemainingSec', 'home_on_ice', 'away_on_ice', 'home_goalie', 'away_goalie', 'merge_idx', 'eventId', 'timeInPeriod', 'timeRemaining_api', 'situationCode', 'homeTeamDefendingSide']


,#,Per,strength,Time:Elapsed Game,Event,Description,Time,timeRemaining,timeInPeriodSec,timeRemainingSec,home_on_ice,away_on_ice,home_goalie,away_goalie,merge_idx,eventId,timeInPeriod,timeRemaining_api,situationCode,homeTeamDefendingSide,typeCode,event_api,sortOrder,period,periodType,maxRegulationPeriods,teamId_,losingPlayerId,winningPlayerId,xCoord,yCoord,zoneCode,reason,hittingPlayerId,hitteePlayerId,playerId,shotType,shootingPlayerId,goalieInNetId,awaySOG,homeSOG,blockingPlayerId,pptReplayUrl,scoringPlayerId,scoringPlayerTotal,assist1PlayerId,assist1PlayerTotal,assist2PlayerId,assist2PlayerTotal,awayScore,homeScore,highlightClipSharingUrl,highlightClip,secondaryReason,typeCode_1,descKey,duration,committedByPlayerId,drawnByPlayerId,highlightClipSharingUrlFr,highlightClipFr,isHome,eventTeam,html_event,home_on_id,away_on_id,homeGoalie_on_id,awayGoalie_on_id,home_on_full_name,away_on_full_name,homeGoalie_on_full_name,awayGoalie_on_full_name,home_on_count,away_on_count,homeGoalie_on_count,awayGoalie_on_count,n_home_skaters,n_away_skaters,pulled_home,pulled_away,home_strength,away_strength,gameStrength,detailedGameStrength,elapsedTime,player1Id,player2Id,player3Id,player1Name,player2Name,player3Name,gameId,homeTeam,awayTeam,teamId,isGoalie,shift_start_type,start_dot,start_zone,venue,venueLocation,gameDate,gameType,startTimeUTC,easternUTCOffset,venueUTCOffset,scrapedOn,source
0,1,1,,0:0020:00,PGSTR,,00:00,20:00,0,1200,[],[],[],[],0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,1,1,<NA>,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,NaN,NaN,NaN,2023020001,TBL,NSH,NaN,NaN,NaN,NaN,NaN,Amalie Arena,Tampa,2023-10-10,2,2023-10-10T21:30:00Z,-04:00,-04:00,2026-03-04T20:02:19.939791,NHL Play-by-Play API
1,2,1,,0:0020:00,PGEND,,00:00,20:00,0,1200,[],[],[],[],0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,1,1,<NA>,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,NaN,NaN,NaN,2023020001,TBL,NSH,NaN,NaN,NaN,NaN,NaN,Amalie Arena,Tampa,2023-10-10,2,2023-10-10T21:30:00Z,-04:00,-04:00,2026-03-04T20:02:19.939791,NHL Play-by-Play API
2,3,1,,0:0020:00,ANTHEM,,00:00,20:00,0,1200,[],[],[],[],0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[],[],[],0.0,0.0,0.0,0.0,0.0,0.0,1,1,<NA>,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,NaN,NaN,NaN,2023020001,TBL,NSH,NaN,NaN,NaN,NaN,NaN,Amalie Arena,Tampa,2023-10-10,2,2023-10-10T21:30:00Z,-04:00,-04:00,2026-03-04T20:02:19.939791,NHL Play-by-Play API
3,4,1,,0:0020:00,PSTR,Period Start- Local time: 5:32 EDT,00:00,20:00,0,1200,"[64, 71, 38, 48, 77]","[75, 90, 9, 27, 45]",[31],[74],0.0,102.0,00:00,20:00,1551,left,520.0,period-start,8.0,1,REG,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NSH,PSTR,"[8477353, 8478519, 8479542, 8480246, 8475167]","[8481704, 8475158, 8476887, 8474151, 8478851]",[8477992],[8477424],"[Tyler Motte, Anthony Cirelli, Brandon Hagel, ...","[Juuso Parssinen, Ryan O'Reilly, Filip Forsber...",[Jonas Johansson],[Juuse Saros],5.0,5.0,1.0,1.0,4.0,4.0,0,0,5,5,5v5,5v5,0,<NA>,<NA>,<NA>,NaN,NaN,NaN,2023020001,TBL,NSH,NaN,NaN,NaN,NaN,NaN,Amalie Arena,Tampa,2023-10-10,2,2023-10-10T21:30:00Z,-04:00,-04:00,2026-03-04T20:02:19.939791,NHL Play-by-Play API
4,5,1,NaN,NaN,ON,NaN,0:00,20:00,0,1200,NaN,NaN,NaN,NaN,NaN,NaN,0:00,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,20:00,NaN,NaN,NaN,NaN,1.0,TBL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [60]:
# ── shifts() ──────────────────────────────────────────────────────────────────
try:
    shifts = nhl.shifts(2023020001)
    print('Shape:', shifts.shape)
    display(shifts.head())
except AttributeError:
    print('shifts() not exposed as standalone — shift data is contained in scrape_game()')

Shape: (743, 44)


,shift_number,period,start_time_elapsed_game,end_time_elapsed_game,duration,event,player_name,jersey_number,team_type,team_name,start_time_in_period,start_time_remaining,end_time_in_period,end_time_remaining,duration_seconds,period_number,isHome,teamId,playerId,sweaterNumber,positionCode,headshot,firstName.default,lastName.default,lastName.cs,lastName.fi,lastName.sk,firstName.cs,firstName.de,firstName.es,firstName.fi,firstName.sk,firstName.sv,lastName.sv,fullName,start_time_in_period_seconds,start_time_remaining_seconds,end_time_in_period_seconds,end_time_remaining_seconds,elapsed_time_start,elapsed_time_end,gameId,homeTeam,awayTeam
0,1,1,1:28 / 18:32,2:24 / 17:36,00:56,,"11 GLENDENING, LUKE",11,Home,TAMPA BAY LIGHTNING,1:28,18:32,2:24,17:36,56,1,1,14,8476822,11,C,https://assets.nhle.com/mugs/nhl/20232024/TBL/...,Luke,Glendening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Luke Glendening,88,1112,144,1056,88.0,144.0,2023020001,TBL,NSH
1,2,1,4:52 / 15:08,5:25 / 14:35,00:33,,"11 GLENDENING, LUKE",11,Home,TAMPA BAY LIGHTNING,4:52,15:08,5:25,14:35,33,1,1,14,8476822,11,C,https://assets.nhle.com/mugs/nhl/20232024/TBL/...,Luke,Glendening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Luke Glendening,292,908,325,875,292.0,325.0,2023020001,TBL,NSH
2,3,1,7:54 / 12:06,8:35 / 11:25,00:41,,"11 GLENDENING, LUKE",11,Home,TAMPA BAY LIGHTNING,7:54,12:06,8:35,11:25,41,1,1,14,8476822,11,C,https://assets.nhle.com/mugs/nhl/20232024/TBL/...,Luke,Glendening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Luke Glendening,474,726,515,685,474.0,515.0,2023020001,TBL,NSH
3,4,1,10:06 / 9:54,10:40 / 9:20,00:34,,"11 GLENDENING, LUKE",11,Home,TAMPA BAY LIGHTNING,10:06,9:54,10:40,9:20,34,1,1,14,8476822,11,C,https://assets.nhle.com/mugs/nhl/20232024/TBL/...,Luke,Glendening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Luke Glendening,606,594,640,560,606.0,640.0,2023020001,TBL,NSH
4,5,1,12:05 / 7:55,12:31 / 7:29,00:26,,"11 GLENDENING, LUKE",11,Home,TAMPA BAY LIGHTNING,12:05,7:55,12:31,7:29,26,1,1,14,8476822,11,C,https://assets.nhle.com/mugs/nhl/20232024/TBL/...,Luke,Glendening,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Luke Glendening,725,475,751,449,725.0,751.0,2023020001,TBL,NSH


In [61]:
# ── shift_chart() ─────────────────────────────────────────────────────────────
try:
    sc = nhl.shift_chart(2023020001)
    print('Type:', type(sc))
    if isinstance(sc, pd.DataFrame):
        print('Shape:', sc.shape)
        display(sc.head())
    else:
        print(list(sc.keys())[:10] if isinstance(sc, dict) else sc)
except AttributeError:
    print('shift_chart() not exposed as standalone — use scrape_game()')

shift_chart() not exposed as standalone — use scrape_game()


---
## 14 — Functional API — `scrape()` <a id="14"></a>

Top-level convenience function. A one-liner alternative to instantiating `HockeyScraper` manually.

```python
from scrapernhl import scrape
df = scrape(league, data_type, **kwargs)
```

| Parameter | Description |
|---|---|
| `league` | League code: `'nhl'`, `'ahl'`, `'pwhl'`, `'ohl'`, `'whl'`, `'qmjhl'` |
| `data_type` | What to fetch: `'pbp'`, `'stats'`, `'schedule'`, `'roster'`, `'standings'` |
| `**kwargs` | Forwarded to the corresponding `HockeyScraper` method |

In [56]:
# ── Functional API — scrape() ─────────────────────────────────────────────────
try:
    from scrapernhl import scrape

    pbp_fn       = scrape('ahl', 'pbp',       game_id=1027781)
    standings_fn = scrape('ahl', 'standings')
    stats_fn     = scrape('ahl', 'stats',     season=90, position='skaters')

    print('scrape("ahl", "pbp")       shape:', pbp_fn.shape)
    print('scrape("ahl", "standings") shape:', standings_fn.shape)
    print('scrape("ahl", "stats")     shape:', stats_fn.shape)
except ImportError:
    print('scrape() is not exported at the top level in this version')

scrape("ahl", "pbp")       shape: (61, 65)
scrape("ahl", "standings") shape: (32, 28)
scrape("ahl", "stats")     shape: (1099, 32)


---
## 15 — Strength Notation Reference <a id="15"></a>

The `strength` (or equivalent) column in PBP DataFrames uses the notation below.  
Strength is always reported from the **shooting / event team's perspective**.

| Code | Description |
|---|---|
| `EV` | Even strength |
| `PP` | Power play — team has more skaters on ice |
| `SH` | Shorthanded — team has fewer skaters on ice |
| `5v5` | 5-on-5 |
| `5v4` | 5-on-4 power play |
| `5v3` | 5-on-3 two-man advantage |
| `4v5` | 4-on-5 shorthanded |
| `4v4` | 4-on-4 (coincidental minor penalties) |
| `3v3` | 3-on-3 (NHL regular-season OT) |
| `EN` | Empty net — shooting team's goalie is pulled |
| `EA` | Empty net against — opposing goalie is pulled |

In [57]:
# ── Inspect strength/situation values in a real NHL PBP DataFrame ─────────────
candidates = ['strength', 'situation', 'strengthCode', 'strength_state', 'typeDescKey']
found = [c for c in candidates if c in pbp_nhl.columns]

if found:
    for col in found:
        print(f'\n--- {col} ---')
        print(pbp_nhl[col].value_counts().to_string())
else:
    str_cols = [c for c in pbp_nhl.columns
                if any(k in c.lower() for k in ['strength', 'situation', 'state', 'skater'])]
    print('Strength-like columns found:', str_cols)
    for c in str_cols[:3]:
        print(f'\n--- {c} ---')
        print(pbp_nhl[c].value_counts().to_string())


--- typeDescKey ---
typeDescKey
faceoff            60
shot-on-goal       57
hit                45
stoppage           44
missed-shot        31
blocked-shot       30
takeaway           18
giveaway           15
penalty            10
goal                8
period-start        3
delayed-penalty     3
period-end          3
game-end            1


---
## 16 — CLI Reference <a id="16"></a>

Run `scrapernhl` directly from the terminal — no Python script needed.

```bash
# General usage
scrapernhl <league> <data_type> [options]

# Play-by-play
scrapernhl ahl pbp --game-id 1027781
scrapernhl nhl pbp --game-id 2023020001

# Standings
scrapernhl ahl standings --season 90
scrapernhl nhl standings --season 20232024

# Player stats
scrapernhl ahl stats --season 90 --position skaters
scrapernhl ahl stats --season 90 --position goalies

# Schedule
scrapernhl ahl schedule --season 90
scrapernhl nhl schedule --team MTL --season 20232024

# Roster
scrapernhl ohl roster --team 7
scrapernhl nhl roster --team MTL --season 20232024
```

Output is printed as CSV to stdout by default. Use `--output <file.csv>` to save to disk.

In [58]:
# ── Verify the CLI entry point is installed and working ───────────────────────
import subprocess, sys

r = subprocess.run([sys.executable, '-m', 'scrapernhl', '--help'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout)
else:
    r2 = subprocess.run(['scrapernhl', '--help'], capture_output=True, text=True)
    out = r2.stdout or r2.stderr
    print(out if out else 'CLI entry point not found — check package installation.')

Usage: python -m scrapernhl [OPTIONS] COMMAND [ARGS]...

  ScraperNHL - Command-line interface for multi-league hockey data scraping.

  Scrape NHL, PWHL, AHL, OHL, WHL, and QMJHL data including teams, schedules,
  standings, rosters, stats, and games directly from the command line.

Options:
  --version  Show the version and exit.
  --help     Show this message and exit.

Commands:
  ahl        AHL (American Hockey League) commands.
  draft      Scrape NHL draft data.
  game       Scrape play-by-play data for a specific game.
  ohl        OHL (Ontario Hockey League) commands.
  pwhl       PWHL (Professional Women's Hockey League) commands.
  qmjhl      QMJHL (Quebec Maritimes Junior Hockey League) commands.
  roster     Scrape team roster.
  schedule   Scrape team schedule.
  standings  Scrape NHL standings.
  stats      Scrape team player statistics.
  teams      Scrape all NHL teams.
  whl        WHL (Western Hockey League) commands.

